# Busca de threshold para recall de AVC ≥ 95%

Este notebook é exploratório: **não altera o threshold do projeto**. Ele carrega o pipeline salvo, calcula probabilidades out-of-fold somente no conjunto de treino e percorre os thresholds em um `for`. Entre os thresholds que atingem recall mínimo de 95% para `stroke=1`, seleciona aquele com maior precisão.

Por fim, aplica uma única vez o threshold candidato ao holdout reservado e mostra se a meta se sustenta. O holdout não participa da busca. Como há poucos positivos, o resultado pode variar bastante entre amostras; avalie também a quantidade de falsos positivos e não interprete o threshold como recomendação clínica.

Execute antes `python -m src.models.train` para gerar `models/stroke_pipeline.joblib`.

In [1]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, fbeta_score,
                             precision_score, recall_score)
from config.paths_config import DATA_PATH, MODELS_DIR
from config.training_config import CV_FOLDS, RANDOM_STATE, TEST_SIZE
from src.data.loader import load_dataset
from src.features.engineering import split_features_target

TARGET_RECALL = 0.95
bundle = joblib.load(MODELS_DIR / 'stroke_pipeline.joblib')
df = load_dataset(DATA_PATH)
X, y = split_features_target(df)
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
print(f"Modelo: {bundle['model_name']} + {bundle['sampler']} | positivos no treino: {int(y_train.sum())} | positivos no holdout: {int(y_holdout.sum())}")

ModuleNotFoundError: No module named 'config'

## Probabilidades out-of-fold no treino

Cada pessoa do treino recebe uma probabilidade produzida por um modelo que não foi treinado com aquela linha. A reamostragem SMOTENC continua dentro do pipeline e é refeita somente nos dados de treino de cada fold.

In [ ]:
oof_probabilities = cross_val_predict(
    clone(bundle['pipeline']), X_train, y_train, cv=cv,
    method='predict_proba', n_jobs=1
)[:, 1]

# Inclui 0 e 1 para cobrir os extremos; a decisão positiva usa >=, igual à API.
thresholds = np.unique(np.r_[0.0, oof_probabilities, 1.0])
rows = []
for threshold in thresholds:
    predictions = (oof_probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_train, predictions, labels=[0, 1]).ravel()
    recall = recall_score(y_train, predictions, zero_division=0)
    precision = precision_score(y_train, predictions, zero_division=0)
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    rows.append({
        'threshold': float(threshold), 'recall_stroke_1': recall,
        'precision': precision, 'specificity': specificity,
        'f2': fbeta_score(y_train, predictions, beta=2, zero_division=0),
        'true_positives': int(tp), 'false_negatives': int(fn),
        'false_positives': int(fp), 'true_negatives': int(tn),
        'predicted_positive': int(predictions.sum())
    })

threshold_table = pd.DataFrame(rows)
feasible = threshold_table[threshold_table['recall_stroke_1'] >= TARGET_RECALL]
if feasible.empty:
    raise RuntimeError('Nenhum threshold atingiu o recall mínimo no OOF; confira as probabilidades e os dados.')
# Prioriza precisão, e usa especificidade como desempate entre thresholds viáveis.
candidate = feasible.sort_values(['precision', 'specificity', 'threshold'], ascending=False).iloc[0]
candidate_threshold = float(candidate['threshold'])
print(f"Threshold candidato para recall ≥ {TARGET_RECALL:.0%}: {candidate_threshold:.6f}")
display(feasible.sort_values(['precision', 'specificity'], ascending=False).head(15).round(4))

## Trade-off por threshold

A linha vertical marca o candidato. A linha horizontal indica o recall mínimo solicitado. Observe como precisão e especificidade mudam junto com o recall.

In [ ]:
curve = threshold_table.sort_values('threshold')
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(curve['threshold'], curve['recall_stroke_1'], label='Recall AVC (1)', linewidth=2)
ax.plot(curve['threshold'], curve['precision'], label='Precisão', linewidth=2)
ax.plot(curve['threshold'], curve['specificity'], label='Especificidade', linewidth=2)
ax.axhline(TARGET_RECALL, color='black', linestyle='--', label='Recall alvo = 95%')
ax.axvline(candidate_threshold, color='red', linestyle=':', label=f'Threshold candidato = {candidate_threshold:.3f}')
ax.set(xlabel='Threshold', ylabel='Métrica', ylim=(0, 1.02), title='Trade-off de métricas no treino out-of-fold')
ax.legend(loc='best'); ax.grid(alpha=0.2); fig.tight_layout()

## Verificação única no holdout

Esta célula mede o candidato no holdout sem reajustar threshold. Se o recall ficar abaixo de 95%, a meta não generalizou para essa amostra. Não escolha outro threshold olhando o holdout; isso transformaria o conjunto de teste em dado de ajuste.

In [ ]:
holdout_probabilities = bundle['pipeline'].predict_proba(X_holdout)[:, 1]
holdout_predictions = (holdout_probabilities >= candidate_threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_holdout, holdout_predictions, labels=[0, 1]).ravel()
holdout_metrics = pd.Series({
    'threshold candidato': candidate_threshold,
    'recall AVC (sensibilidade)': recall_score(y_holdout, holdout_predictions, zero_division=0),
    'precision': precision_score(y_holdout, holdout_predictions, zero_division=0),
    'specificity': tn / (tn + fp) if (tn + fp) else 0.0,
    'F2-score': fbeta_score(y_holdout, holdout_predictions, beta=2, zero_division=0),
    'F1-score': f1_score(y_holdout, holdout_predictions, zero_division=0),
    'accuracy': accuracy_score(y_holdout, holdout_predictions),
    'verdadeiros positivos': tp, 'falsos negativos (AVCs perdidos)': fn,
    'falsos positivos': fp, 'verdadeiros negativos': tn,
    'total sinalizado como positivo': int(holdout_predictions.sum())
}, name='resultado')
display(holdout_metrics.to_frame().round(4))
print('Matriz de confusão [[TN, FP], [FN, TP]]:')
print(confusion_matrix(y_holdout, holdout_predictions, labels=[0, 1]))

## Snapshot da última execução (2026-09-29)

Com o modelo salvo nesta execução, o loop escolheu threshold **0,051919**: no treino out-of-fold, recall **0,960** (191/199 AVCs) e precisão **0,093**. Aplicado sem ajuste ao holdout, obteve recall **0,900** (45/50 AVCs), perdendo 5; precisão **0,088**, com 466 falsos positivos; matriz `[[506, 466], [5, 45]]`. Portanto, o candidato **não confirmou 95% no holdout** e geraria muitos alarmes. Rode novamente as células após um novo treino para recalcular o snapshot.